# 🎙️ Sokhan — realtime voice assistant in Colab

Talk to a fully local **VAD → STT → LLM → TTS** assistant right from your browser: streaming replies,
interruptions (barge-in), 5-second voice cloning, and optional webcam vision.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (optional — the CPU works too, just slower).

Run the cells from top to bottom. The live voice UI appears in step 5.

## 1 · Get the code

In [ ]:
#@title Clone or upload Sokhan { display-mode: "form" }
REPO_URL = "https://github.com/mr-r0ot/Sokhan-Omini-Light"  #@param {type:"string"}
import os, subprocess
if not os.path.isdir("sokhan"):
    if os.path.isdir("Sokhan-Omini-Light/sokhan"):
        %cd Sokhan-Omini-Light
    elif "mr-r0ot" not in REPO_URL:
        !git clone -q {REPO_URL} Sokhan-Omini-Light
        %cd Sokhan-Omini-Light
    else:
        raise SystemExit("Set REPO_URL to your repository, or upload the project folder (with sokhan/ inside) "
                         "to the Colab file browser and run this cell again.")
print("Sokhan source:", os.getcwd())

## 2 · Install (≈2–5 min)

In [ ]:
#@title Install dependencies { display-mode: "form" }
import os, subprocess
GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
!pip -q install sherpa-onnx onnxruntime onnx soundfile sentencepiece huggingface_hub psutil pillow
if GPU:
    # prebuilt CUDA wheel when one exists; otherwise pip builds it with CUDA (slower, one time)
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    !pip -q install "llama-cpp-python>=0.3.16" --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
else:
    !pip -q install "llama-cpp-python>=0.3.16" --prefer-binary --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu
import llama_cpp
print("GPU:", GPU, "| llama-cpp-python", llama_cpp.__version__)

## 3 · Configure

Everything has sensible defaults (4-bit models). Change the language and the speech models follow:
`fa` uses Shenava STT + Pocket-TTS (voice cloning), other languages use Whisper + a Piper voice.

In [ ]:
#@title Settings { display-mode: "form" }
LANGUAGE = "fa"  #@param ["fa", "en", "de", "fr", "es", "ar", "tr"]
LLM_MODEL = "unsloth/Qwen3.5-4B-GGUF"  #@param {type:"string"}
GREETING = ""  #@param {type:"string"}
ENABLE_CAMERA = False  #@param {type:"boolean"}

import logging
from sokhan import Config, Omni
logging.basicConfig(level=logging.WARNING)

cfg = Config.for_language(LANGUAGE)
cfg.llm.model = LLM_MODEL
cfg.hardware.use_gpu = GPU            # falls back to CPU automatically
cfg.prompt.greeting = GREETING
if ENABLE_CAMERA:
    cfg.vision.enabled = True         # Qwen3.5 ships a vision projector; it is downloaded too

## 4 · Load the models (first run downloads them)

In [ ]:
import ipywidgets as w
from IPython.display import display
bar, label = w.FloatProgress(min=0, max=1, layout=w.Layout(width="60%")), w.Label("starting…")
display(w.VBox([label, bar]))

omni = Omni(cfg)
def progress(stage, fraction, message):
    bar.value, label.value = fraction, f"{stage}: {message}"
omni.on("load_progress", progress)
omni.start()
label.value = "✅ ready"
print("capabilities:", omni.capabilities)
print("plan:", omni.plan)

Quick text check (no audio):

In [ ]:
print(omni.ask("Say hello in one short sentence."))

## 5 · Talk 🎙️

Press **Start talking**, allow the microphone, and speak. Talk over it any time to interrupt.
Keep this the last running cell — the audio flows while the notebook is idle.

In [ ]:
from sokhan.colab import ColabBridge
bridge = ColabBridge(omni, title="Sokhan", camera=ENABLE_CAMERA).show()

## 6 · Clone a voice (optional)

Upload a clean **3–5 second** recording (WAV/FLAC/MP3). Only runs when the voice model supports cloning.

In [ ]:
if omni.capabilities["voice_cloning"]:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        omni.clone_voice(name)
        omni.say("سلام! این صدای جدید منه." if cfg.language == "fa" else "Hi! This is my new voice.",
                 remember=False)
        print("✓ voice cloned from", name)
else:
    print("The current voice model cannot clone voices.")

## 7 · Tools (optional)
Give the assistant functions it can call:

In [ ]:
import datetime
from sokhan import tool

@tool
def current_time() -> str:
    """The current date and time."""
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

omni.add_tool(current_time)
print(omni.ask("What time is it?"))

## 8 · Finish

In [ ]:
bridge.close()
omni.close()